In [ ]:
# Add Parquet do elasticsearch index
import pyarrow.parquet as pq
import polars as pl
from elasticsearch import Elasticsearch, helpers

PARQUET_PATH = "data/iowa_data/iowa_sales_sentiments.parquet"
INDEX_NAME = "iowa_liquor_sales_reviews_2"

READ_BATCH_SIZE = 1000_000
BULK_CHUNK_SIZE = 5_000

es = Elasticsearch(
    "http://localhost:9200",
    request_timeout=120,
    retry_on_timeout=True,
    max_retries=3
)

pf = pq.ParquetFile(PARQUET_PATH)


def actions_from_polars(df: pl.DataFrame):
    for row in df.iter_rows(named=True):
        yield {
            "_op_type": "index",
            "_index": INDEX_NAME,
            "_source": row
        }


total_loaded = 0

for arrow_batch in pf.iter_batches(batch_size=READ_BATCH_SIZE):
    df = pl.from_arrow(arrow_batch)

    ok_count = 0
    fail_count = 0

    for ok, result in helpers.streaming_bulk(
        client=es,
        actions=actions_from_polars(df),
        chunk_size=BULK_CHUNK_SIZE,
        request_timeout=120,
        raise_on_error=False,
        raise_on_exception=False
    ):
        if ok:
            ok_count += 1
        else:
            fail_count += 1
            print("FAILED:", result)

    total_loaded += ok_count

    print(
        f"Batch loaded: {ok_count:,}, failed: {fail_count:,}, total: {total_loaded:,}"
    )

print(f"Done. Total loaded: {total_loaded:,}")

In [1]:
from elasticsearch import Elasticsearch
from pprint import pprint

INDEX_NAME = "iowa_liquor_sales_reviews_2"

es = Elasticsearch("http://localhost:9200")

print(es.info())

{'name': 'efee423d087c', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'cA_w3MTnQQObd-a6dGAq7A', 'version': {'number': '9.4.2', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': 'c402c2b36d90eae29c0182f86bd9050fd0b746cc', 'build_date': '2026-05-25T22:10:36.017759931Z', 'build_snapshot': False, 'lucene_version': '10.4.0', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'}


In [2]:
query = {
    "size": 5,
    "query": {
        "match": {
            "all_review_text": "tequila"
        }
    }
}

response = es.search(index=INDEX_NAME, body=query)

for hit in response["hits"]["hits"]:
    print("Score:", hit["_score"])
    print(hit["_source"]["all_review_text"])
    print("-" * 80)
print(response)

Score: 2.6568353
Customers considered this silver tequila a reliable purchase because of the smooth finish. The delivery window was reliable for the liquor order. The price supported repeat purchases. Service quality matched regular expectations, affecting how buyers rated the transaction.
--------------------------------------------------------------------------------
Score: 2.6568353
Shoppers were disappointed with the silver tequila, mainly because of its weak aroma. The delivery window was reliable for the liquor order. The price supported repeat purchases. Store staff handled questions professionally, affecting how buyers rated the transaction.
--------------------------------------------------------------------------------
Score: 2.6568353
Shoppers were disappointed with the reposado tequila, mainly because of its rough aftertaste. The delivery window was reliable for the liquor order. The price supported repeat purchases. Staff explained bottle options clearly, affecting how buy

In [4]:
query = {
    "size": 10,
    "query": {
        "multi_match": {
            "query": "teqila botle qualty",
            "fields": [
                "product_review_text^3",
                "all_review_text"
            ],
            "fuzziness": "AUTO"
        }
    },
    "highlight": {
        "fields": {
            "product_review_text": {},
            "all_review_text": {}
        }
    }
}

response = es.search(index=INDEX_NAME, body=query)

for hit in response["hits"]["hits"]:
    print("Score:", hit["_score"])
    print(hit["_source"]["all_review_text"])
    print(hit.get("highlight"))
    print("-" * 80)

Score: 15.895648
Shoppers gave positive feedback on this reposado tequila, especially because of its consistent bottle quality and dependable quality. Bottle cases were delivered cleanly and without damage, which influenced the store's view of the purchase cycle. Buyers felt the quality justified the retail price during the purchase decision. The feedback was recorded after the transaction. Customers reported poor assistance during selection while customers selected bottles from the shelf.
{'all_review_text': ['Shoppers gave positive feedback on this reposado <em>tequila</em>, especially because of its consistent <em>bottle</em>', '<em>quality</em> and dependable <em>quality</em>.', "<em>Bottle</em> cases were delivered cleanly and without damage, which influenced the store's view of the purchase", 'Buyers felt the <em>quality</em> justified the retail price during the purchase decision.'], 'product_review_text': ['Shoppers gave positive feedback on this reposado <em>tequila</em>, espe

In [5]:
query = {
    "size": 0,
    "aggs": {
        "sentiments": {
            "terms": {
                "field": "overall_sentiment.keyword",
                "size": 3
            },
            "aggs": {
                "total_sales": {
                    "sum": {
                        "field": "SaleDollars"
                    }
                }
            }
        }
    }
}

response = es.search(index=INDEX_NAME, body=query)

for bucket in response["aggregations"]["sentiments"]["buckets"]:
    print(
        bucket["key"],
        bucket["doc_count"],
        round(bucket["total_sales"]["value"], 2)
    )

positive 30242217 4490174248.89
neutral 2148099 319100168.39
negative 867672 128216468.34
